# Implementing a simple Retrieval-augmented generation (RAG)
This tutorial is based on a workshop created by our friends of the [Swiss-AI-Center](https://www.hes-so.ch/swiss-ai-center).

Original Authors:
- Jonathan Guerne, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Henrique Marques Reis, Research assistant at Haute Ecole Arc Ingénierie, Switzerland
- Célien Donzé, Scientific Collaborator at HEIA-FR, Switzerland

Adapted by:
- Francesco Carrino, Professor at Haute Ecole Ingénierie, Valais/Wallis Switzerland

## Introduction
This notebook explains how to create a RAG (Retrieval Augmented Generation) system to answer questions about online or PDF documents. We will use a self-hosted LLM with Ollama to generate answers to the questions. Additionally, we will use a vector database to retrieve relevant documents for answering the questions.

Technologies and tools used in this exercise:
- [Large Language Model (LLM)](https://en.wikipedia.org/wiki/Large_language_model), a model that generates text (typically) baseded on the transformer architecture. In this lab, we will used opensource models like LLAMA 3.x.
- [Ollama](https://ollama.com/): Ollama is a platform that democratizes access to LLMs by enabling users to run them locally on their machines. In this lab, it will be used to run the LLMs on your machine or in the cloud (Kaggle).
- [Vector database](https://en.wikipedia.org/wiki/Vector_database): A Vector Database is a relational database system specifically designed to process vectorized data. Unlike conventional databases that contain information in tables, rows, and columns, vector databases work with vectors–arrays of numerical values that signify points in multidimensional space. When used with text data, vector databases can store and retrieve information based on the semantic meaning of the text, rather than just the text itself. Through a process called embedding, text data is converted into vectors that represent the **meaning** of the text. This meanse text with similar meaning is stored close to each other in the vector space. We will use [FAISS](https://faiss.ai/) an opensource library for efficient similarity search and clustering of dense vectors. In this lab, it will be used to store information about pdf and web documents.
- [LangChain](https://python.langchain.com/docs/introduction/): LangChain is a framework for developing applications powered by large language models (LLMs).  It is based on the idea of "chains", where a chain defines sequences of actions, where each step can involve querying an LLM, prompt management, manipulating data, or interacting with external tools applications. In this lab, it will be used to put together the LLM and the vector database to create a RAG system.
- [Retrieval-augmented generation (RAG)](https://en.wikipedia.org/wiki/Retrieval-augmented_generation): RAG is a technique to improve LLMs by adding a retrieval component from an external data source. This is the main goal of this tutorial.

### Structure
This lab is divided into 4 parts. The first 3 parts, you will be guided to implement a simple RAG working on pdf file. The last part is a challenge where you can try to implement a RAG working on web pages.
1. **Installation**: Install the necessary libraries and download the data. We provide two guides: one for running the code on Kaggle (or Google Colab), and another for running the code locally.
2. **Creating and manipulating prompts**: We will create prompts and creatge simple query the LLM and the vector database.
3. **Creating a RAG system**: We will create a simple RAG system to answer questions about pdf documents. The pdf will be embedded in the vector database and the LLM will generate the answers.
4. **Challenge**: In this last part, *you* will implement a RAG system to answer questions about web pages.

---

## 1. Installation



If your machine is not very performant, you can use Kaggle or Google Colab to run this notebook. The following code block installs the necessary packages.
If you choose to run the code on Kaggle, login to your account and import this notebook. It is raccomanded to have a verified account to have access to the GPU.

Below you can find the instraction to install the necessary packages either on Kaggle or on your local machine.

### Installing Ollama (Kaggle or Google Colab)

If you are in Kaggle or Google Colab, you can install Ollama using the following code block. If want to run this notebook locally, see the next section.

These instructions are based on the Medium post titled [How to Run Ollama Models on Google Colab and Kaggle: A Complete Guide](https://medium.com/@mradul18varshney/how-to-run-ollama-generative-ai-models-on-google-colab-and-kaggle-a-complete-guide-3e01fc12512f).

In [ ]:
# Package installation (Kaggle or Google Colab)
!pip install langchain==0.2.7 langchain-community faiss-cpu pymupdf pypdf sentence_transformers rich wget python-dotenv cryptography

#### Setting up the environment (Kaggle or Google Colab)

This update package list and install curl, which then use to fetch Ollama installation script.

In [ ]:
# Update system files and install curl
!apt-get update && apt-get install -y curl

#### Installing Ollama (Kaggle or Google Colab)

This will install Ollama automatically.

In [ ]:
# Install Ollama
!curl https://ollama.ai/install.sh | sh

#### Starting the Ollama server (Kaggle or Google Colab)

Here, we use Python’s `subprocess.Popen()` to run the Ollama server in the background, listening on the specified host and port (127.0.0.1:11438). The server will be available for communication via the API.

You will need this address later for the RAG model.

In [ ]:
import subprocess
import time

# Set the environment variable for the Ollama host and port
os.environ['OLLAMA_HOST'] = '127.0.0.1:11438'

# Start the Ollama server in a subprocess
ollama_process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Wait for the server to start
print("Starting Ollama server...")
time.sleep(1)  # Wait for the server to initialize

print("Ollama server is running on 127.0.0.1:11438")

#### Pulling an open-source LLM model (Kaggle or Google Colab)
This command downloads the Llama 3.2 model from Ollama’s servers, making it available for use on your notebook machine environment.
Also, refer here https://ollama.com/search for checking the list of models available on Ollama model repository.

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', 'llama3.2'])

#### Interacting with the model (Kaggle or Google Colab)

To test the installation and the model, we can use the following code block to interact with the model.

In [ ]:
import requests
import json

def generate_response(query):
    # Define the API URL for the same host as mentioned when starting the server.
    url = "http://localhost:11438/api/chat"
    
    # Define the payload (data) to send in the POST request
    payload = {
        "model": "llama3.2",
        "messages": [
            {
                "role": "user",
                "content": query
            }
        ],
        "keep_alive": 0,
        "stream": False
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    # Make the POST request to the API
    try:
        response = requests.post(url, data=json.dumps(payload), headers=headers)
    
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            output = response.text
            return output
        else:
            print(f"Failed to get a response. Status code: {response.status_code}")
            return ("Error response:", response.text)
    except Exception as e:
        print(f"An error occurred: {e}")

Once the server is running and the model is pulled, you can start interacting with the model using Python code. In this example, the generate_response() function sends a POST request to the Ollama API, passing in the user query. The model processes the query and returns a generated response.

*Note: If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull. Also, check the model name to match with the model you pulled using `ollama pull` command.*

In [ ]:
# Using the function to get the output
# The output is in JSON format, and you can parse it using Python’s json module.

query = "What do you know about the new google chip willow?"
output = generate_response(query)
json.loads(output)['message']['content']

#### Using Ollama’s Python SDK (Kaggle or Google Colab)

To interact with the Ollama server, you can use the Python SDK. The SDK is available on PyPI and can be installed using pip.

In [ ]:
!pip install ollama

The code below interacts with the Ollama model using the Python SDK to send a query and print the generated response.

*Note: (again) If you change the model you will get the error that the model is not downloaded and you have to pull it using ollama pull.*

In [ ]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='llama3.2', messages=[
  {
    'role': 'user',
    'content': 'Show me the lyrics of any song?',
  },
])
print(response['message']['content'])

If all works, you should see the generated response from the model. In my test, I got the lyrics from the song "Yesterday" by The Beatles. What did you get?

- - - 

### Installing Ollama (locally on your machine)

This guide will help you to install Ollama on your local machine. You need a fairly good machine to run LLMs models and space in your hard disk to store the LLM models (several GB).  If you have a machine with a GPU, you can run the models faster. However, there are lighter models that can run on CPU and provide results good enough for this lab and many applications (e.g., you can try "qwen2.5:0.5b" model).

#### Download and install Ollama
Ollma is available for macOS, Linux, and Windows. Go to the official [website](https://ollama.com/) and download the version for your operating system. Follow the instructions to install it.

#### Running Ollama
After installing Ollama, you should see the Ollama icon in your system tray. You can use it to see the logs and quit the application.

Usuall, after the installation Ollama is already running. 

If you want to run it again, you can run the following command in your terminal:

```bash
ollama serve
```

To run Ollama, go to the [Ollama model list](https://ollama.com/search) and select a model suitable for your machine. Start humble with a small model. You can for instance try the small version of [LLAMA 3.2](https://ollama.com/library/llama3.2).

Follow the instractions for the model you selected. For example, for LLAMA 3.2 (the 1B parameters version), run in your terminal:

```
ollama run llama3.2:1b
```

The `run` command will pull (download) the model to your machine and then run it, exposing it via the API started with `ollama serve`.

If you want to *pull* a differen model, you can run:

```bash
ollama pull <model_name>
```

The commanda `ollama list` will show you the list of models available on your machine.

To switch between models available on your machine, you can just use the `run` command (NOTE: if the model is not present in your machine, the `run` command will autamtically try to pull it). The `stop` command will stop a running model.

#### Get Ollama address
In order to interact with the model, you need to know the address where the model is running. Ollama binds to the local address 127.0.0.1 on port 11434 by default.

This means that you can note http://localhost:11434

### Package installation (with Poetry)
We will install the necessary packages using Poetry. We provide a `pyproject.toml` file with the necessary dependencies. We assume you have [Poetry installed](https://github.com/hei-synd-aml/lab-0-TutoPoetry) on your machine. Open a terminal in the project location and run the following command to install the packages.

```bash
poetry install
```

---

## 2. Creating a prompt and interacting with the model

### Imports

In [2]:
import os
from pathlib import Path

import langchain
import wget
from dotenv import load_dotenv
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain.document_loaders import PyPDFDirectoryLoader
# from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_community.llms.ollama import Ollama
from langchain_core.documents.base import Document
from rich.console import Console
from rich.markdown import Markdown

console = Console()

### Documentation

- [langchain](https://python.langchain.com/v0.1/docs/get_started/introduction/)
- [Ollama website](https://ollama.com/)

### Constants

In [3]:
load_dotenv(override=True)

# Kaggle
# OLLAMA_ADDRESS = "http://localhost:11438"  # replace with your OLLAMA address
#LLM_NAME = "llama3.2" # replace with the LLM name you pulled from the OLLAMA

# Local Ollama
OLLAMA_ADDRESS = "http://localhost:11434"  # replace with your OLLAMA address, usually http://localhost:11434
LLM_NAME = "llama3.2:1b"  # replace with the LLM name you pulled from the OLLAMA


### Connecting to LLM

NOTE: if you get "ConnectionError", check if the Ollama server is running and the address is correct.

In [5]:
llm = Ollama(
    model=LLM_NAME,
    base_url=OLLAMA_ADDRESS,
    temperature=0.1,  # Will be explained later
    stop=["<end_of_turn>"],
)

llm.generate(["Hello, how are you today?"]).generations[0][0].text

"I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help you with any questions or topics you'd like to discuss. How about you? How's your day going so far?"

### Creating a prompt

A prompt is generally divided into two parts: the context and the question.

The context provides the information that the model will use to generate its answer, while the question specifies what the model is expected to respond to.

Additionally, LangChain requires markers indicating where to insert the user's question and the context retrieved from documents.

[Langchain prompt templates documentation](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates)

In [6]:
template = """
You are an helpful assistant that answer the question in detail.

Human input: {question}
Assistant:"""

prompt = PromptTemplate(input_variables=["question"], template=template)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an helpful assistant that answer the question in detail.\n\nHuman input: {question}\nAssistant:')

### Creating the chain and start a conversation

In [5]:
llm_chain = prompt | llm

In [6]:
result = llm_chain.invoke(input="When is the AI-days 2025?")

console.print(Markdown(result))

I can provide you with information on the topic of "AI days" or "Artificial Intelligence Days," but I must clarify 
that there isn't a specific event called "AI days" that has been officially announced by any organization.         

However, I can tell you about some notable events and milestones related to artificial intelligence that might be  
of interest:                                                                                                       

 • 2011: The first public demonstration of a deep learning algorithm was presented at the International Joint      
   Conference on Artificial Intelligence (IJCAI).                                                                  
 • 2012: IBM's Watson system defeated human champions in several trivia games, including Jeopardy!, which helped   
   establish it as one of the most powerful AI systems.                                                            
 • 2014: Google announced its AlphaGo project, a deep learning-based artificial intelligence system that aimed to  
   defeat world chess champions.                                                                                   
 • 2016: The European Union launched the Horizon 2020 program, which included several initiatives focused on AI    
   research and development.                                                                                       

As for specific dates related to "AI days," I couldn't find any information about a particular event or celebration
being held in 2025. However, there are several conferences and events that might be of interest:                   

 • The International Conference on Machine Learning (ICML) is an annual conference that focuses on machine learning
   and AI research.                                                                                                
 • The International Joint Conference on Artificial Intelligence (IJCAI) is another prominent conference that      
   covers various aspects of artificial intelligence.                                                              
 • The World Economic Forum's (WEF) Global Future Councils often host events and discussions related to AI, which  
   might be of interest.                                                                                           

If you could provide more context or clarify what you mean by "AI days" 2025, I'd be happy to try and help further.

---

## 3. Using the RAG

For the moment, we have a model that can generate answers to questions. To provide answers, it only uses its knolwege got at training time and the context we provide. But what if we could provide the model with a set of documents and ask it to retrieve the most relevant ones to answer the question?
That is the idea behind the Retrieval-Augmented Generation (i.e., literally, we augment the generation of text by enriching the context with information contained in relevant documents).

### Downloading the pdfs

In [7]:
# Create the "data/PDFs" folder if it doesn't exist
PDF_FOLDER = Path("data/PDFs")
os.makedirs(PDF_FOLDER, exist_ok=True)

urls = [
    "https://ai-days.swiss-ai-center.ch/PDF/Session1.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session2b.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3a.pdf",
    "https://ai-days.swiss-ai-center.ch/PDF/Session3b.pdf",
]

# Download the PDFs
for url in urls:
    name = url.split("/")[-1]
    if not (PDF_FOLDER / name).is_file():
        filename = wget.download(url, f"data/PDFs/{name}")
console.print("Pdf file downloaded successfully.", style="bold green")

Pdf file downloaded successfully.

### Loading a PDF

In [8]:
VECTORSTORES_DIR = Path("data/vectorstores")
PDF_FOLDER

WindowsPath('data/PDFs')

In [9]:
loader = PyPDFDirectoryLoader(PDF_FOLDER)
doc = loader.load()
doc

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong po

[Document(metadata={'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20250116093703Z00'00'", 'title': 'Workshop_descriptions', 'moddate': "D:20250116093703Z00'00'", 'source': 'data\\PDFs\\Session1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY 27\nSession 1: Compute Infrastructures for IA applications in the wild Location: Movie theater 6 With the advent of Chatbots, LLMs and other generative IA technologies, as well as other progresses in the IA ﬁeld, there is an explosion of the demand for compute force. IA is no longer computer science: it is computational science. As such, it can no longer be done with casual, self-managed equipment. More advanced compute infrastructures are required both to satisfy user needs (in terms of compute power, GPU Ram capacity) and to ensure a decent utilization of the increasingly costly

### Embedding a PDF in a vectorstore

In [10]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5" # locally you can also use smaller models. Check the HuggingFace model hub for more models with different sizes, performance and languages: https://huggingface.co/spaces/mteb/leaderboard

In [11]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    # to use the "cuda" configuration, you need an nvidia GPU, and to install 
    # On Kaggle, you set to use as accelrator GPU P100 (you need a verified account)
    # model_kwargs={"device": "cuda"}, # change to cuda -> cpu if you do not have an nvdia GPU
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

c:\Users\ingno\AppData\Local\pypoetry\Cache\virtualenvs\lab-04-rag-K-_h4Q_l-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
all_splits = text_splitter.split_documents(doc)
vectorstore = FAISS.from_documents(documents=all_splits, embedding=embedding_model)

In [13]:
vectorstore.save_local(VECTORSTORES_DIR)

### Loading a vectorstore

In [14]:
vectorstore = FAISS.load_local(
    VECTORSTORES_DIR, embedding_model, allow_dangerous_deserialization=True
)

#### NOTE: What is temperature?

The temperature parameter in a language model (LLM) controls the randomness of the model's output.

A lower temperature value (closer to 0) makes the model more deterministic, favoring higher probability words and resulting in more predictable and repetitive text.

A higher temperature value (closer to 1) increases randomness, allowing for more creative and diverse responses by giving less probable words a better chance of being chosen.

Adjusting the temperature helps balance between coherence and creativity in the generated text.

### New prompt

In RAG we need to add another marker to indicate where the new information (or context) should be inserted for this we use the variable named `{context}`.

In [15]:
prompt = """
Use the following pieces of context to answer the question at the end.
Don't try to make up an answer and only use the information you know.
Use three sentences maximum and keep the answer as concise as possible.
You must answer in english.
Context:
{context}

Question:
{input}

Answer:
"""

prompt_template = PromptTemplate(input_variables=["context", "input"], template=prompt)
prompt_template

PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\nUse the following pieces of context to answer the question at the end.\nDon't try to make up an answer and only use the information you know.\nUse three sentences maximum and keep the answer as concise as possible.\nYou must answer in english.\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n")

Tips and tricks: You can modifiy the prompte above to get an answer closer to the one you are looking for.

### Creating the chain

In [16]:
# Top k of chunks to retrieve from the vectorstore
NB_RETRIVED_CHUNKS = 8

In [17]:
question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt_template)
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": NB_RETRIVED_CHUNKS,
    }
)
chain = create_retrieval_chain(retriever, question_answer_chain)

### Chatting with a pdf

In [18]:
result = chain.invoke(input={"input": "When is the AI-days 2025?"})

console.print(Markdown(result["answer"]))

The AI-DAYS@HES-SO 2025 event took place from January 27 to January 29, 2025.

### Explaining the output

The `result` dictionary contains much more than the simple answer. Thanks to the metadata, we can see the title of the document, the page number, and the context where the answer was found!

In [19]:
console.print(result)

{
    'input': 'When is the AI-days 2025?',
    'context': [
        Document(
            id='88fe62ad-85ad-4251-b9ae-93e01a006ae1',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 0,
                'page_label': '1'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='d9a3a70e-f086-4b03-be9a-fdc61f647659',
            metadata={
                'producer': 'macOS Version 15.1.1 (Build 24B91) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250116093703Z00'00'",
                'title': 'Workshop_descriptions',
                'moddate': "D:20250116093703Z00'00'",
                'source': 'data\\PDFs\\Session1.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='0903b9ab-d603-4662-babb-d312296cc075',
            metadata={
                'producer': 'macOS Version 15.0.1 (Build 24A348) Quartz PDFContext, AppendMode 1.1',
                'creator': 'Word',
                'creationdate': "D:20241105080725Z00'00'",
                'moddate': "D:20241105080730Z00'00'",
                'title': 'Workshop_descriptions',
                'source': 'data\\PDFs\\Session2a.pdf',
                'total_pages': 2,
                'page': 0,
                'page_label': '1'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='5eca7e6d-7102-435f-926a-6d898119c673',
            metadata={
                'producer': 'macOS Version 15.0.1 (Build 24A348) Quartz PDFContext, AppendMode 1.1',
                'creator': 'Word',
                'creationdate': "D:20241105080725Z00'00'",
                'moddate': "D:20241105080730Z00'00'",
                'title': 'Workshop_descriptions',
                'source': 'data\\PDFs\\Session2a.pdf',
                'total_pages': 2,
                'page': 1,
                'page_label': '2'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='cd01dc04-3938-48c2-a612-eb8f08581bb4',
            metadata={
                'producer': 'macOS Version 15.2 (Build 24C101) Quartz PDFContext',
                'creator': 'Word',
                'creationdate': "D:20250118184043Z00'00'",
                'title': 'session2b',
                'moddate': "D:20250118184043Z00'00'",
                'source': 'data\\PDFs\\Session2b.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
            },
            page_content='AI-DAYS@HES-SO 2025 –GENEVA & LAUSANNE –JANUARY 27-JANUARY 29, 2025\nWORKSHOP DAY JANUARY
27'
        ),
        Document(
            id='cb26a3cd-f994-42f6-8cb0-4fafd9cfbc07',
            metadata={
                'producer': 'macOS Version 15.0.1 (Build 24A348) Quartz PDFContext, AppendMode 1.1',
                'creator': 'Word',
                'creationdate': "D:20241105080804Z00'00'",
                'moddate': "D:20241105080809Z00'00'",
                'title': 'Workshop_descriptions',
                'source': 'data\\PDFs\\Session3a.pdf',
                'total_pages': 1,
                'page': 0,
                'page_label': '1'
     

---

## 4. Challenge: Getting information from a website

Now, it is your turn. Use the code above as inspiration to:
- interact with a website (instead of a PDF)
- create a vectorstore with the information from the website
- chat with the model
- evaluate the output


We suggest to use https://docs.realgames.co/homeio/en/, but you can choose any website you want. We *expect* that your code works with any "normal" website.
